In [13]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from tqdm import tqdm

class StockDataset(Dataset):
    def __init__(self, data, seq_len=30):
        self.data = data
        self.seq_len = seq_len
        
    def __len__(self):
        return len(self.data) - self.seq_len
    
    def __getitem__(self, idx):
        x = self.data[idx:idx+self.seq_len]
        y = self.data[idx+1:idx+self.seq_len+1]  # 偏移一位作为目标值
        return torch.FloatTensor(x), torch.FloatTensor(y)

class Transformer(nn.Module):
    def __init__(self, input_size=30, d_model=512, nhead=8, num_encoder_layers=6,
                 num_decoder_layers=6, output_size=1):
        super(Transformer, self).__init__()
        
        # 编码器
        self.encoder_input_layer = nn.Linear(input_size, d_model)
        self.encoder_layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead) 
            for _ in range(num_encoder_layers)])
        # 解码器
        self.decoder_layers = nn.ModuleList([
            nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead)
            for _ in range(num_decoder_layers)])
        self.decoder_output_layer = nn.Linear(d_model, output_size)
        
    def forward(self, x):
        # 编码器输入：x shape [seq_len, batch_size, input_size]
        enc_input = x.permute(1, 0, 2)  # 转换为[batch_size, seq_len, d_model]
        enc_out = self.encoder_input_layer(enc_input)
        for layer in self.encoder_layers:
            enc_out = layer(enc_out)
        
        # 解码器输入：x shape [seq_len, batch_size, d_model]
        dec_input = x.permute(1, 0, 2)  # 转换为[batch_size, seq_len, d_model]
        for layer in self.decoder_layers:
            dec_out = layer(dec_input, enc_out)
        dec_out = self.decoder_output_layer(dec_out.permute(1, 0, 2))
        
        return dec_out

def get_technical_indicators(data):
    # data: DataFrame with 'Open', 'High', 'Low', 'Close', 'Volume'
    indicators = []
    
    # 计算移动平均线（MA）
    ma_5 = data['Close'].rolling(5).mean()
    ma_10 = data['Close'].rolling(10).mean()
    
    # 计算波动率（Variance）
    volatility = data['Close'].rolling(5).var()
    
    # 计算相对强指数（RSI）
    delta = data['Close'] - data['Close'].shift(1)
    gain = delta.where(delta >= 0, 0)
    loss = delta.where(delta < 0, 0)
    avg_gain = gain.rolling(5).mean()
    avg_loss = loss.rolling(5).mean()
    rsi = 100 - (avg_loss / avg_gain) * 100
    
    indicators.append(ma_5)
    indicators.append(ma_10)
    indicators.append(volatility)
    indicators.append(rsi)
    
    # 将所有指标合并到dataframe
    tech_df = pd.DataFrame(indicators).T
    return tech_df

def main():
    # 加载股票数据
    data = pd.read_csv('stock_data.csv')  # 假设你有一个名为'stock_data.csv'的文件
    
    # 提取技术指标特征
    tech_indicators = get_technical_indicators(data)
    
    # 创建Dataset和DataLoader
    seq_len = 30  # 序列长度
    dataset = StockDataset(tech_indicators.values, seq_len=seq_len)
    batch_size = 64
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # 初始化模型
    input_size = seq_len  # 每个样本有30个时间步，每个时间步有多个特征
    d_model = 512  # embedding dimension
    nhead = 8     # number of attention heads
    num_encoder_layers = 6
    num_decoder_layers = 6
    output_size = 1
    
    model = Transformer(input_size=input_size, d_model=d_model,
                       nhead=nhead, num_encoder_layers=num_encoder_layers,
                       num_decoder_layers=num_decoder_layers, output_size=output_size)
    
    # 定义损失函数和优化器
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    # 训练模型
    num_epochs = 100
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        
        for batch_idx, (x, y) in enumerate(dataloader):
            optimizer.zero_grad()
            outputs = model(x.permute(1, 0, 2))  # 转换为[seq_len, batch_size, input_size]
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch: {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")
    
    # 保存模型
    torch.save(model.state_dict(), 'transformer_stock_model.pth')

if __name__ == '__main__':
    main()

FileNotFoundError: [Errno 2] No such file or directory: 'stock_data.csv'